# SEA-Net - data analysis

A thin, read-only notebook. All the figures and tables are built by **`seanet/report.py`** (the same code `python main.py report` runs), so the plotting lives in one place, not here.

This notebook just calls it and shows the results. It trains nothing; run it any time to see whatever has finished so far. Everything is also saved under `results/SEA_NET/`.

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root (the folder that contains the "seanet" package) and make it importable.
root = Path.cwd()
while root != root.parent and not (root / "seanet" / "__init__.py").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

from seanet import report
from IPython.display import Image, display
print("repo root:", root)

## 1. Generate everything

This draws every figure and writes the summary table + the SEA-Net vs MILLET comparison.

In [ ]:
# Same call as `python main.py report`. Prints the headline numbers and saves the figures.
out = report.generate_report(verbose=True)

## 2. The figures

Shown from the PNGs just saved under `results/SEA_NET/figures/`. In the scatter plots, a point **above the red y=x line means SEA-Net wins** on that dataset.

In [ ]:
figdir = report.FIGURES_DIR
for name in ["data_summary.png", "results.png", "acc_scatter.png",
             "win_tie_loss.png", "aopcr_scatter.png", "acc_diff.png"]:
    path = os.path.join(figdir, name)
    if os.path.exists(path):
        display(Image(filename=path))

## 3. The underlying tables

The data behind the figures: SEA-Net's per-dataset results, and the comparison to MILLET (only the ~85 datasets that overlap the paper have a MILLET number).

In [ ]:
summary, results, comparison = report.load_frames()
print("datasets summarised:", len(summary),
      "| results:", len(results),
      "| comparison rows:", len(comparison))
display(results[["dataset", "test_acc", "test_aopcr", "test_ndcg", "train_time_s"]].head())
overlap = comparison[comparison["acc_outcome"] != "no_baseline"]
display(overlap[["dataset", "ours_acc", "millet_acc", "acc_outcome",
                 "ours_aopcr", "millet_aopcr", "aopcr_outcome"]].head(10))

## Notes

- **Accuracy** vs MILLET: the first scatter, the win/tie/loss bar, and the per-dataset gap.
- **AOPCR** (interpretation faithfulness): the second scatter. Most points sit below the line, so SEA-Net's explanations are less faithful than MILLET's - the main thing to improve.
- To regenerate outside the notebook: `python main.py report`.
- Re-run after more of the sweep finishes to refresh every figure.